# HOT-NERD usage demo
This notebook demonstrates reading the example CSVs in the `examples/` folder and simple usages of the `hotnerd` package.

In [1]:
%pip install -e ..

Obtaining file:///Users/lyuan13/Desktop/HOT-NERD
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hotnerd (pyproject.toml) ... done
  Created wheel for hotnerd: filename=hotnerd-0.1.0-0.editable-py3-none-any.whl size=3955 sha256=df37b69352378b414438c269b37cd116efae852c9cd6dc130903231f552e7974
  Stored in directory: /private/var/folders/_2/50fjjk3d1696gqy3jz2ypb9j88m1t6/T/pip-ephem-wheel-cache-z3d6dgn6/wheels/d5/ff/3d/4bcd2e7f2e50cbb52fb70788614108d2f3277469b1a77b8326
Successfully built hotnerd
  Attempting uninstall: hotnerd
    Found existing installation: hotnerd 0.1.0
    Uninstalling hotnerd-0.1.0:
      Successfully uninstalled hotnerd-0.1.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

def find_repo_root(start=Path.cwd(), max_up=6):
    p = start.resolve()
    for _ in range(max_up):
        if (p / "pyproject.toml").exists() or (p / "setup.py").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return start.resolve()

repo_root = find_repo_root()
sys.path.insert(0, str(repo_root / "src"))
print("Added to sys.path:", sys.path[0])

Added to sys.path: /Users/lyuan13/Desktop/HOT-NERD/src


In [3]:
try:
    import hotnerd
    print("hotnerd", hotnerd.__version__)
except Exception as e:
    print("import failed:", e)

/Users/lyuan13/anaconda3/envs/spatial/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


hotnerd 0.1.0


In [4]:
# Common notebook imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')
%matplotlib inline

In [5]:
p = repo_root / "examples"   
expected = ['example_transcripts.csv', 'example_npmi_long.csv', 'example_cell_geometries.csv']

# list files and report missing
found = {f.name: f for f in p.glob('*.csv')}
for fn in expected:
    print(fn, '->', found.get(fn, 'MISSING'))

# read into dict of DataFrames (guarded)
dfs = {}
for fn in expected:
    fp = found.get(fn)
    if fp is None:
        dfs[fn] = pd.DataFrame()
        continue
    try:
        dfs[fn] = pd.read_csv(fp)
    except Exception as e:
        print(f"Failed to read {fn}:", e)
        dfs[fn] = pd.DataFrame()

# assign convenient names
df_transcripts = dfs['example_transcripts.csv']
df_npmi = dfs['example_npmi_long.csv']
df_geoms = dfs['example_cell_geometries.csv']

# quick heads
print(df_transcripts.shape); print(df_transcripts.head())
print(df_npmi.shape); print(df_npmi.head())
print(df_geoms.shape); print(df_geoms.head())

example_transcripts.csv -> /Users/lyuan13/Desktop/HOT-NERD/examples/example_transcripts.csv
example_npmi_long.csv -> /Users/lyuan13/Desktop/HOT-NERD/examples/example_npmi_long.csv
example_cell_geometries.csv -> /Users/lyuan13/Desktop/HOT-NERD/examples/example_cell_geometries.csv
(100219, 9)
           x          y          z feature_name  cell_id  overlaps_nucleus  \
0  5487.9893  1273.5900  28.554290          LUM   124682                 0   
1  5489.9004  1276.8323  31.048160          LUM   124682                 0   
2  5493.7095  1302.3413  29.888750        RUNX1   124692                 0   
3  5495.0860  1289.5330  26.718008         LPXN   124692                 0   
4  5495.5576  1326.9908  27.053818        ITGAX   124712                 1   

     transcript_id    qv                                    geometry  
0  281535106257056  40.0  POINT (5487.9892578125 1273.5899658203125)  
1  281535106257092  40.0    POINT (5489.900390625 1276.832275390625)  
2  281535106257150  40.0  

In [6]:
# ensure repo root cell has been run so `hotnerd` is importable
from hotnerd import prune_transcripts

# df_transcripts and df_npmi are loaded earlier in the notebook
# df_transcripts must contain your transcript rows with a cell id column (default "cell_id")
# df_npmi must be long-form with columns like "gene_i", "gene_j", "NPMI"

df_pruned, aux = prune_transcripts(
    df=df_transcripts,
    npmi_df=df_npmi,
    cell_id_col="cell_id",
    gene_col="feature_name",
    threshold=-0.2,      # adjust as needed
    unassigned_id="-1",  # string used for partial ids, e.g. "cell_id-1"
)

# inspect results
print(df_pruned.shape)
display(df_pruned.head())
print("aux keys:", list(aux.keys()))

(100219, 13)


,x,y,z,feature_name,cell_id,overlaps_nucleus,transcript_id,qv,geometry,cell_id_npmi_cons_p1,npmi_cons_p1_status,cell_id_npmi_cons_p2,npmi_cons_p2_status
0,5487.9893,1273.5900,28.554290,LUM,124682,0,281535106257056,40.0,POINT (5487.9892578125 1273.5899658203125),124682,core,124682,unchanged
1,5489.9004,1276.8323,31.048160,LUM,124682,0,281535106257092,40.0,POINT (5489.900390625 1276.832275390625),124682,core,124682,unchanged
2,5493.7095,1302.3413,29.888750,RUNX1,124692,0,281535106257150,40.0,POINT (5493.70947265625 1302.34130859375),124692,core,124692,unchanged
3,5495.0860,1289.5330,26.718008,LPXN,124692,0,281535106257175,40.0,POINT (5495.0859375 1289.532958984375),124692,core,124692,unchanged
4,5495.5576,1326.9908,27.053818,ITGAX,124712,1,281535106257187,40.0,POINT (5495.5576171875 1326.9908447265625),124712-1,partial_p1,-1,unassigned_from_partial


aux keys: ['genes', 'gene_to_idx', 'W', 'partial_map', 'threshold']


In [7]:
from hotnerd import diagnostic_npmi_report
# or: import hotnerd; diagnostic_npmi_report = hotnerd.diagnostic_npmi_report

# pick an example cell id (ensure types match: str vs int)
cell_id_example = df_transcripts.loc[df_transcripts["cell_id"] != -1, "cell_id"].iloc[0]

# call the diagnostic and show result
diag_one = diagnostic_npmi_report(
    df=df_pruned,
    aux=aux,
    cell_id=cell_id_example,
)

print(type(diag_one))      # should be a pandas.DataFrame
display(diag_one)          # or print(diag_one.head())

<class 'pandas.core.frame.DataFrame'>


,stage,n_transcripts,n_unique_genes,sum_npmi,min_npmi,p25,p50,p75,n_pairs
0,original,159,63,-42.808006,-0.521276,-0.151085,-0.015709,0.091282,1919
1,core_pass1,105,41,33.970737,-0.197009,-0.048652,0.018799,0.096297,799
2,partial_pass1,54,22,-4.879949,-0.521276,-0.336235,-0.016560,0.268790,230
3,partial_pass2,43,14,26.816736,0.017372,0.205266,0.278643,0.358003,91
4,unassigned_from_partial,11,8,1.975944,-0.275986,-0.169584,0.038389,0.307367,28


In [8]:
# import functions
from hotnerd import annotate_unassigned_components, build_graph, prune_genes_by_npmi_greedy

# call (uses df_pruned and aux returned from prune_transcripts)
df_final = annotate_unassigned_components(
    df_pruned=df_pruned,
    aux=aux,
    build_graph_fn=build_graph,                      # hotnerd.build_graph or your own
    prune_fn=prune_genes_by_npmi_greedy,             # hotnerd.prune_genes_by_npmi_greedy
    coord_cols=("x", "y", "z"),
    k=5,
    dist_threshold=1.5,
    min_comp_size=15,
    npmi_threshold=-0.1,
    unassigned_final_col="cell_id_npmi_cons_p2",
    cell_id_col="cell_id",
    gene_col="feature_name",
    transcript_id_col="transcript_id",
)

# inspect results
print(df_final.shape)
display(df_final.head())
print(df_final["unassigned_qc_status"].value_counts(dropna=False))

Constructed 45,297 edges among 23,278 transcripts (k≤5, d≤1.5 µm)
(100219, 17)


,x,y,z,feature_name,cell_id,overlaps_nucleus,transcript_id,qv,geometry,cell_id_npmi_cons_p1,npmi_cons_p1_status,cell_id_npmi_cons_p2,npmi_cons_p2_status,unassigned_comp_id,unassigned_qc_status,unassigned_comp_size,cell_id_final
0,5487.9893,1273.5900,28.554290,LUM,124682,0,281535106257056,40.0,POINT (5487.9892578125 1273.5899658203125),124682,core,124682,unchanged,NaN,NaN,NaN,124682
1,5489.9004,1276.8323,31.048160,LUM,124682,0,281535106257092,40.0,POINT (5489.900390625 1276.832275390625),124682,core,124682,unchanged,NaN,NaN,NaN,124682
2,5493.7095,1302.3413,29.888750,RUNX1,124692,0,281535106257150,40.0,POINT (5493.70947265625 1302.34130859375),124692,core,124692,unchanged,NaN,NaN,NaN,124692
3,5495.0860,1289.5330,26.718008,LPXN,124692,0,281535106257175,40.0,POINT (5495.0859375 1289.532958984375),124692,core,124692,unchanged,NaN,NaN,NaN,124692
4,5495.5576,1326.9908,27.053818,ITGAX,124712,1,281535106257187,40.0,POINT (5495.5576171875 1326.9908447265625),124712-1,partial_p1,-1,unassigned_from_partial,UNASSIGNED_0,drop_small_comp,1.0,DROP


unassigned_qc_status
NaN                      76941
drop_small_comp          15507
keep_unassigned_comp      5301
drop_npmi_pruned_gene     2470
Name: count, dtype: int64


In [9]:
from hotnerd import apply_stitching_to_transcripts
# or: import hotnerd; apply_stitching_to_transcripts = hotnerd.apply_stitching_to_transcripts

# call (uses `df_final` and `aux` from earlier steps)
df_stitched, entity_to_stitched = apply_stitching_to_transcripts(
    df_final=df_final,
    aux=aux,
    entity_col="cell_id_final",
    gene_col="feature_name",
    coord_cols=("x", "y", "z"),
    purity_threshold=0.05,
    penalize_simplicity=True,
    deltaC_min=0.0,
    use_3d=True,
    out_col="cell_id_stitched",
)

# quick checks
print(df_stitched.shape)
display(df_stitched.head())
print("stitched mapping sample:", dict(list(entity_to_stitched.items())[:10]))

(100219, 18)


,x,y,z,feature_name,cell_id,overlaps_nucleus,transcript_id,qv,geometry,cell_id_npmi_cons_p1,npmi_cons_p1_status,cell_id_npmi_cons_p2,npmi_cons_p2_status,unassigned_comp_id,unassigned_qc_status,unassigned_comp_size,cell_id_final,cell_id_stitched
0,5487.9893,1273.5900,28.554290,LUM,124682,0,281535106257056,40.0,POINT (5487.9892578125 1273.5899658203125),124682,core,124682,unchanged,NaN,NaN,NaN,124682,124682
1,5489.9004,1276.8323,31.048160,LUM,124682,0,281535106257092,40.0,POINT (5489.900390625 1276.832275390625),124682,core,124682,unchanged,NaN,NaN,NaN,124682,124682
2,5493.7095,1302.3413,29.888750,RUNX1,124692,0,281535106257150,40.0,POINT (5493.70947265625 1302.34130859375),124692,core,124692,unchanged,NaN,NaN,NaN,124692,124692
3,5495.0860,1289.5330,26.718008,LPXN,124692,0,281535106257175,40.0,POINT (5495.0859375 1289.532958984375),124692,core,124692,unchanged,NaN,NaN,NaN,124692,124692
4,5495.5576,1326.9908,27.053818,ITGAX,124712,1,281535106257187,40.0,POINT (5495.5576171875 1326.9908447265625),124712-1,partial_p1,-1,unassigned_from_partial,UNASSIGNED_0,drop_small_comp,1.0,DROP,DROP


stitched mapping sample: {'11185': '11185', '11185-1': '11185-1', '11195': '11195', '11195-1': '11195-1', '124162': '124162', '124162-1': '124162-1', '124163': '124163', '124163-1': '124163-1', '124164': '124164', '124164-1': '124164-1'}


In [10]:
from hotnerd import plot_3d_concave_cell, plot_3d_convex_cell

png = plot_3d_convex_cell(
    df_stitched,
    cell_id=124838,
    out_png="cell_124838_3d_comparison.png",
)

print("Saved:", png)

Saved: cell_124838_3d_comparison.png


In [13]:
png = plot_3d_concave_cell(
    df_stitched,
    cell_id=124838,
    k=18,
    scale=1.5,
    out_png="cell_124838_concave_refinement.png"
)

print("Saved:", png)

Saved: cell_124838_concave_refinement.png
